In [0]:
from __future__ import annotations

import datetime as dt
import functools
import json
import os
from typing import Any, Callable, Iterable, List, Optional

import data.utilities.common as Commonlib

import numpy as np
import pandas as pd

from beertech_utils.core.logger import LogManager as LM
from beertech_utils.data.df.pyspark_utils import PySparkUtils as PU
from beertech_utils.data.integrity import Quarantine
from beertech_utils.data.integrity.integrity_check import (
    CompletenessIntegrityCheck,
    DuplicateIntegrityCheck,
    EquivalencyIntegrityCheck,
)
from data.utilities.compression import Zip

from data.utilities.extractors import SFTPExtractor, SnowflakeExtractor
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU

from data.utilities.publishers.ddl_gen import DDL
from pyspark.dbutils import DBUtils
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import (
    col,
    expr,
    lit,
    monotonically_increasing_id,
    regexp_replace,
    row_number,
    split,
    to_date,
    trim,
    when,
)
from pyspark.sql.types import (
    BooleanType,
    DateType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.utils import AnalysisException
from pyspark.sql.window import Window

ALCHEMY_KEYVAULT = "alchemy-kv"
PY_DTYPES = {
    "string": str,
    "int": int,
    "float": float,
    "double": float,
    "boolean": bool,
}


def with_logging(func):
    """
    Decorator function that adds logging to Medallion methods.
    Parameters
    ----------
    func: function
        The function passed in to the decorator
    Returns
    -------
    inner: function
        The decorated function
    """

    @functools.wraps(func)
    def inner(*args, **kwargs):
        self = args[0]
        try:
            ret = func(*args, **kwargs)
            self.logger.info(f"Step {func.__name__} completed successfully for {self.name}")
            return ret
        except Exception as e:
            self.logger.error(e)
            raise e

    return inner


def enforce_types(on_error: str = "warn") -> Callable:
    """Enforce type annotations on method

    Args:
        on_error (str): how to handle mismatched type. Default: "warn".
            "warn" will produce a logger warning with information on the type mismatch
            "error" will throw a type error

    Returns:
        Callable: method that does type checking

    Throws:
        TypeError: if there is a mismatch for an argument and `on_error` is "error"
    """

    def deco(func):
        def inner(self, *args, **kwargs):
            if func.__annotations__:
                # all functions have __annotations__ but returns {} if there aren't any
                for idx, (arg_name, annotation_type) in enumerate(func.__annotations__.items()):
                    try:
                        if args and len(args) > idx:
                            arg = args[idx]
                            assert isinstance(arg, annotation_type)
                        elif kwargs and arg_name in kwargs:
                            arg = kwargs[arg_name]
                            assert isinstance(arg, annotation_type)
                    except AssertionError as e:
                        message = (
                            # calling type(arg) works inconsistently, can't figure out why
                            # my first guess is that it has something to do with the timing
                            # of the error handling.
                            # it doesn't break, just shows up blank, printing out the value still
                            # works though so this error message is still valuable
                            f"Function {func.__name__} argument {arg_name} is {type(arg)}({arg})"
                            f" instead of specified {annotation_type}"
                        )
                        if on_error == "warn":
                            self.logger.warning(message)
                        elif on_error == "error":
                            self.logger.error(f"{message}: {e}")
                            raise TypeError(f"{message}: {e}")

            return func(self, *args, **kwargs)

        return inner

    return deco


def enforce_member_type(name: str, _type: type, on_error: str = "warn") -> Callable:
    """Makes an assertion about a member of self

    Args:
        name (str): class member to check
        _type (type): type to enforce

    Returns:
        Callable: method that enforces member type before execution

    Throws:
        TypeError: if there is a type mismatch between specification and reality
    """

    def deco(func):
        def inner(self, *args, **kwargs):
            try:
                assert isinstance(getattr(self, name), _type)
            except AssertionError:
                message = f"Member {name} of {self} does not match asserted type: {_type}"
                if on_error == "warn":
                    self.logger.warning(message)
                elif on_error == "error":
                    self.logger.error(message)
                    raise TypeError(message)
            return func(self, *args, **kwargs)

        return inner

    return deco


class Medallion:
    DECIMAL_DEFAULT_PRECISION = 12
    DECIMAL_DEFAULT_SCALE = 2
    SFTP_EXTRACTOR = "sftp"
    SFTP_GET_FILES_BY_PATTERN = "get_files_by_pattern"
    SFTP_OPERATIONS_SUPPORTED = [SFTP_GET_FILES_BY_PATTERN]

    def __init__(self, config: dict):
        self.name = config.get("name")
        self.schema = config.get("schema")
        self.catalog = config.get("catalog")
        self.description = config.get("description")
        self.silver_name = config.get("silver_name")
        self.subject = config.get("subject")
        self.region = config.get("region")
        self.src_location = config.get("src_location")
        self.src_filename = config.get("src_filename")
        self.src_format = config.get("src_format")
        self.start_datetime = dt.datetime.utcnow()
        self.bronze_schema = config.get("bronze", {}).get("schema")
        self.silver_schema = config.get("silver", {}).get("schema")
        self.gold_schema = config.get("gold", {}).get("schema")
        self.source_config = config.get("source", {})
        self.source_config["raw_path"] = (
            f"dbfs:/mnt/raw-internal{self.source_config.get('raw_path')}"
            if self.source_config.get("raw_path")
            else None
        )
        try:
            self.key_columns = [
                column["name"] for column in self.silver_schema if column.get("metadata", dict()).get("key", False)
            ]
        except Exception:
            self.key_columns = [
                column["name"] for column in self.gold_schema if column.get("metadata", dict()).get("key", False)
            ]
        try:
            self.update_check_columns = [
                column["name"]
                for column in self.silver_schema
                if (column["name"] not in self.key_columns) & (column["name"][:2] != "__")
            ]
            self.update_condition = " or ".join(
                [f"source.{column} != target.{column}" for column in self.update_check_columns]
            )
        except Exception:
            self.update_check_columns = [
                column["name"]
                for column in self.gold_schema
                if (column["name"] not in self.key_columns) & (column["name"][:2] != "__")
            ]
            self.update_condition = " or ".join(
                [f"source.{column} != target.{column}" for column in self.update_check_columns]
            )
        self.df = None
        self.watermarks = {}
        self.watermark_lookup = {
            "bronze": config.get("bronze", {}).get("watermark", None),
            "silver": config.get("silver", {}).get("watermark", None),
        }
        self.table_lookup = {
            "bronze": MTU.get_fully_qualified_table_name(self.catalog, self.schema, self.name, "bronze"),
            "silver": MTU.get_fully_qualified_table_name(self.catalog, self.schema, self.name),
            "gold": MTU.get_fully_qualified_table_name(self.catalog, self.schema, self.name),
        }
        self.split_table_lookup = {"bronze": {}, "silver": {}, "gold": {}}
        self.spark = SparkSession.builder.getOrCreate()
        self.dbutils = DBUtils(self.spark)
        self.logger = MTU.configure_datadog_logging(
            add_tags={
                "medallion_name": self.name,
                "object_runtime": dt.datetime.now(),
            }
        )
        self.default_watermark = config.get("default_watermark", dt.datetime(1900, 1, 1))

        for layer in [m.value for m in ML]:
            if config.get(layer, None):
                table_name = self.table_lookup[layer]
                (
                    self.split_table_lookup[layer]["catalog"],
                    self.split_table_lookup[layer]["schema"],
                    self.split_table_lookup[layer]["table"],
                ) = table_name.split(".")

    @with_logging
    def list_raw_files(
        self,
        *,
        sort_on: str = "modificationTime",
        sort_order: str = "desc",
        exclude_extensions: List[str] | None = None,
    ) -> "dbutils.FileInfo":
        """Lists all the raw files in the medallion's raw path., if self.src_filename is not set, otherwise filters on self.src_filename.
        also exclude's zip files

        Args:
            sort_on (str): file sort type. Default: "modificationTime". Options: {"modificationTime", "name"}
            sort_order (str): sorting order. Default: "desc". Options: {"asc", "desc"}
            exclude_extensions (list<str>): file type extensions to exclude i.e. "zip"

        None
        Returns
        -------
        List[FileInfo]: A list of raw file names that match the provided conditions.
        """
        # empty string `in` any other string evaluates to True so if
        # no src file name is set; default to "" so that all files are returned
        file_pattern = self.src_filename if self.src_filename else ""
        excludes = (
            exclude_extensions
            if not isinstance(exclude_extensions, str) and isinstance(exclude_extensions, Iterable)
            else tuple()
        )

        return list(
            filter(
                # filters for file pattern
                # and if exlude_extensions is defined then it will filter all file extensions in exlude
                lambda x: file_pattern in x.name
                and not x.isDir()
                and not (exclude_extensions is not None and (x.split(".")[-1] not in excludes)),
                sorted(
                    self.dbutils.fs.ls(self.source_config.get("raw_path")),
                    key=(lambda x: getattr(x, sort_on)),
                    reverse=(sort_order == "desc"),
                ),
            )
        )

    @with_logging
    def extract_to_raw(self) -> None:
        """Handles extracting files from a remote location given the objects config dictionary

        Parameters:
            None
        Returns:
            Nothing
        """
        # check if extraction is needed prior to loading
        if Medallion.SFTP_EXTRACTOR in self.source_config:
            # get connection details
            connection_key = self.source_config.get(Medallion.SFTP_EXTRACTOR).get("connection")
            connection_info = json.loads(self.dbutils.secrets.get(scope=ALCHEMY_KEYVAULT, key=connection_key))

            self.logger.info(f"extraction started. connection={connection_key}")

            # perform extractor supported operations
            for operation in Medallion.SFTP_OPERATIONS_SUPPORTED:
                if operation == Medallion.SFTP_GET_FILES_BY_PATTERN:
                    operation_options = {"local": f"{self.source_config.get('raw_path')}"}
                    operation_options = {
                        **operation_options,
                        **self.source_config.get(Medallion.SFTP_EXTRACTOR).get(Medallion.SFTP_GET_FILES_BY_PATTERN),
                    }

                    current_datetime = dt.datetime.now()
                    current_datetime = current_datetime.strftime("_%Y-%m-%dT%H%M%S_%f")

                    # decompress files if set
                    if operation_options.get("decompress_callback") == "zip":
                        operation_options["decompress_callback"] = Zip(move_to="compressed/", suffix=current_datetime)

                    with SFTPExtractor(logger=self.logger, **connection_info) as source:
                        source.get_files_by_pattern(**operation_options)

                    self.logger.info(
                        f"extraction finished. connection={connection_key}, file_pattern={operation_options.get('pattern')}"
                    )
                else:
                    raise NotImplementedError(
                        "extract_to_path: could not find configuration for the supported SFTP operation. skipping."
                    )
        # add more condiftion for other extractors if needed
        else:
            self.logger.info("extract_to_path: extraction options not found in object config.")

    @with_logging
    def read_raw_data(
        self, filename: str, *, file_format: Optional[str] = None, read_options: Optional[dict] = None
    ) -> None:
        """
        Reads a dataframe from the medallion object's raw path.
        Parameters
        ----------
        filename : string
            The name of the file to read from the raw layer
        file_format : string
            The file format you would like to handle within this method. Supports CSV and FIXED.
        read_options : dict
            A dictionary of options that will be passed:
            CSV Options - Sparks "DataFrameReader" Options - https://spark.apache.org/docs/latest/sql-data-sources-csv.html
            FIXED Options - Pandas read_fwf(**kwds) - https://pandas.pydata.org/docs/reference/api/pandas.read_fwf.html
            XML Options - Sparks XML Options - https://github.com/databricks/spark-xml
            EXCEL Options - https://github.com/crealytics/spark-excel#create-a-dataframe-from-an-excel-file
        Returns
        -------
        None
        """
        file_location = os.path.join(self.source_config.get("raw_path"), filename)
        file_format = file_format if file_format else self.src_format
        read_options = read_options if read_options else dict()

        # both `createDataFrame` and `CSVReader.load()` take an optional parameter
        # `schema` where the default is None. If adding a new format in the future
        # that doesn't allow this default behavior you will need to check if spark_schema
        # is None
        spark_schema = None
        if self.bronze_schema:
            # build spark schema, this could/should be done in init but fine for now
            # set defaults for StructType creation - fromJson() method requires all attributes be set
            assert isinstance(self.bronze_schema, list)
            for field in self.bronze_schema:
                field.setdefault("nullable", True)
                field.setdefault("metadata", dict())

            spark_schema = StructType.fromJson({"fields": self.bronze_schema})

        if file_format.upper() == "CSV" or file_format.upper() == "XML" or file_format.upper() == "EXCEL":
            # the `inferSchema` option is overridden by providing a schema so if schema is None
            # it will default to the set option but if schema is provided, the `inferSchema` option
            # is ignored
            # https://spark.apache.org/docs/latest/api/python/_modules/pyspark/sql/readwriter.html#DataFrameReader.csv
            self.df = (
                self.spark.read.format(file_format).options(**read_options).load(file_location, schema=spark_schema)
            )
        elif file_format.upper() == "FIXED":
            file_location = Commonlib.modify_dbfs_path(f"{file_location}")

            # get field lengths and column names
            col_names = [field["name"] for field in self.bronze_schema]
            col_lengths = [field["metadata"]["length"] for field in self.bronze_schema]

            # read fixed width file into pandas df
            pd_df = pd.read_fwf(file_location, widths=col_lengths, names=col_names, dtype=str, **read_options)
            pd_df.replace({np.nan: None}, inplace=True)

            self.df = self.spark.createDataFrame(pd_df, schema=spark_schema)
        else:
            raise AttributeError(f"read_raw_data does not support files of type {file_format}")

    @with_logging
    def remove_first_n_rows(self, n: int = 1) -> None:
        """
        Removes the first row of the medallion's dataframe.
        Parameters
        ----------
        None
        Returns
        -------
        None
        """
        self.df = (
            # monotonically increasing id guarantees increasing by partition
            # currently - this respects the original file order
            self.df.withColumn("_index", monotonically_increasing_id())
            .withColumn("__RANK", row_number().over(Window.orderBy(col("_index"))))
            .filter(col("__RANK") > n)
            .drop("_index")
            .drop("__RANK")
        )

    @with_logging
    def remove_last_n_rows(self, n: int = 1):
        """
        Removes the last row of the medallion's dataframe.
        Parameters
        ----------
        None
        Returns
        -------
        None
        """
        self.df = (
            # monotonically increasing id guarantees increasing by partition
            # currently - this respects the original file order
            self.df.withColumn("_index", monotonically_increasing_id())
            .withColumn("__RANK", row_number().over(Window.orderBy(col("_index"))))
            .filter(col("__RANK") <= (self.df.count() - n))
            .drop("_index")
            .drop("__RANK")
        )

    @with_logging
    def cast_all_columns_to_string(self):
        """
        Casts all of the medallion's columns to strings.
        Parameters
        ----------
        None
        Returns
        -------
        None
        """
        self.df = self.df.select(*(col(column).cast("string") for column in self.df.columns))

    @with_logging
    @enforce_member_type("silver_schema", list)
    def cast_silver_dtypes_expr(self):
        """Same as cast_silver_dtypes but expects and expression in the config

        Use this method if you need to change names, change types, add constants, computed columns, ect.

        Args:
            None
        Returns:
            None
        """
        # Sanitize column names before casting dtypes
        for colname in list(self.df.columns):
            special_characters = ["/"]  # Special characters to remove
            if colname[0] in special_characters:  # Remove beginning special characters
                new_colname = colname[1:]
            if colname[-1] in special_characters:  # Remove ending special characters
                new_colname = colname[:-1]
            if (colname[0] not in special_characters) & (colname[:-1] not in special_characters):
                new_colname = colname
            for char in new_colname:  # Replace remaining special characters with a "_".
                if char in special_characters:
                    new_colname = new_colname.replace(char, "_")
            self.df = self.df.withColumnRenamed(colname, new_colname)

        columns = [
            expr(_col.get("metadata", dict()).get("expression", _col.get("name")))
            .cast(_col.get("type", "string"))
            .alias(_col.get("name"))
            for _col in self.silver_schema
        ]
        self.df = self.df.select(columns)

    @with_logging
    def drop_duplicates(self, timestamp_column: str = "__created_tsp", rank_column_name: str = "__RANK"):
        """Drops duplicates from the medallion, keeping the most recent records, re-assigns the resulting dataframe to self

        Args:
            timestamp_column (str): timestamp column name. Default: "__created_tsp"
            rank_column_name (str): ranking column name, this is a temporary column used for filtering
                that is dropped before the dataframe is returned. Please make sure that this does
                not match any existing columns because they will be overwritten. Default: "__RANK"

        Returns:
            None
        """
        if self.watermark_lookup["silver"]:
            timestamp_column = self.watermark_lookup["silver"]
        key_columns = self.key_columns if self.key_columns else list(self.df.columns)
        row_number_spec = Window.partitionBy(key_columns).orderBy(col(timestamp_column).desc())

        # Generates a rank column based on the rank_spec, and filters out duplicate rows,
        # keeping the version with the most recent date (rank 1).
        self.df = (
            self.df.withColumn(rank_column_name, row_number().over(row_number_spec))
            .filter(col(rank_column_name) == 1)
            .drop(rank_column_name)
        )

    @with_logging
    def get_watermark(self, layer: Optional[str] = None, watermark_col: Optional[str] = None):
        """
        Gets the high watermark of the specified medallion layer.
        Parameters
        ----------
        layer : string
            The medallion layer to be read from. Options are 'bronze', 'silver', and 'gold'.
        Returns
        -------
        watermark : datetime
            watermark value retrieved
        """
        # check if table exists, if it does not exist then set to global default watermark
        table = self.table_lookup[layer]
        try:
            if self.spark.catalog.tableExists(table):
                watermark = self.spark.sql(f"select max({watermark_col}) from {table}").collect()[0][0]
                watermark = self.default_watermark if watermark is None else watermark
            else:
                self.logger.info(f"Setting default watermark: {self.default_watermark}")
                watermark = self.default_watermark
        except Exception as e:
            self.logger.error(e)
            raise

        self.watermarks[layer] = watermark
        return watermark

    @with_logging
    def read_table(
        self,
        read_layer: str,
        write_layer: str,
    ):
        """
        Reads data from the specified layer into the medallion.
        Parameters
        ----------
        read_layer : string
            The medallion layer to be read from. Options are 'bronze', 'silver', and 'gold'.
        write_layer : string
            The medallion layer being written to.
        Returns
        -------
        None
        """
        read_args = self.table_lookup[read_layer].split(".")
        watermark_col = self.watermark_lookup[write_layer]

        if watermark_col is not None:
            self.get_watermark(write_layer, watermark_col=watermark_col)

            self.df = MTU.read_table(
                catalog_name=read_args[0],
                schema_name=read_args[1],
                table_name=read_args[2],
                qualify_name=False,  # Name is already qualified in the __init__() function
            ).filter(f"{watermark_col} > '{self.watermarks[write_layer]}'")

        else:
            self.df = MTU.read_table(
                catalog_name=read_args[0],
                schema_name=read_args[1],
                table_name=read_args[2],
                qualify_name=False,  # Name is already qualified in the __init__() function
            )

    @staticmethod
    def _get_scd_type2_columns(schema_def: List[dict]) -> dict:
        get_scd_key_column = lambda key: next((col for col in schema_def if col.get("metadata", dict()).get(key)), "")

        start_tsp_col = get_scd_key_column("start_tsp")
        end_tsp_col = get_scd_key_column("end_tsp")
        active_col = get_scd_key_column("active_flag")

        return Commonlib.filter_dict(
            {
                "start_tsp": start_tsp_col["name"],
                "end_tsp": end_tsp_col["name"],
                "tsp_type": start_tsp_col["type"],
                "active_flag": active_col["name"] if active_col else "",  # active flag is optional
                "max_date": end_tsp_col["metadata"].get("max_date", ""),
                "active_value": active_col["metadata"].get("active_value") if active_col else "",
            },
            excludes=("",),
        )

    @with_logging
    def write_to_delta(
        self,
        load_type=None,
        layer=None,
        source=None,
        update_condition=None,
        delete_condition="false",
    ):
        """
        Writes to a delta table using the specified method.
        load_type: Method used to write to delta. Options are "overwrite", "append", "upsert" or "scd_type_2".
        layer: Layer to be written to (bronze, silver, gold)
        source: Source of the data. Used for tracking data lineage.
        update_condition: Defines a SQL expression to be met before updating a record.
        delete_condition: Defines a SQL expression to be met before deleting a record.
        """
        # Replace the sentinel value for the update condition. If the value is None, then set the default from the self.update_condition attribute.
        if update_condition is None:
            update_condition = self.update_condition

        # Lowercase all columns before writing
        for colname in list(self.df.columns):
            new_colname = colname.lower()
            self.df = self.df.withColumnRenamed(colname, new_colname)

        assert ML[layer]  # ensure valid layer specified
        if layer == ML.silver:
            schema_def = self.silver_schema

        elif layer == ML.gold:
            schema_def = self.gold_schema

        if layer == ML.bronze:
            # Set timestamp before writing to bronze
            self.df = self.df.withColumn("__created_tsp", lit(self.start_datetime).cast(TimestampType()))
            self.df = self.df.withColumn("__source", lit(source).cast(StringType()))
            schema_def = None

        table_name = (
            self.silver_name
            if self.silver_name is not None and layer == ML.silver
            else self.split_table_lookup[layer]["table"]
        )

        # Filter columns and check if the table exists
        if layer != "bronze" and self.spark.catalog.tableExists(self.table_lookup[layer]):
            self.logger.info(f"Filtering columns to match {layer} schema.")
            self.df = self.df[[c["name"] for c in schema_def]]
        # If the table does not exist in dev, then generate the ddl from schema defined in the config file and place in the appropriate location in ddl directory
        elif layer != "bronze" and MTU.get_environment() == "dev":
            try:
                self._create_and_run_sql_file(schema_def)
                self.logger.info(f"Filtering columns to match {layer} schema.")
                self.df = self.df[[c["name"] for c in schema_def]]
            except Exception as e:
                self.logger.error(e)
                raise

        MTU.write_table(
            df=self.df,
            catalog_name=self.split_table_lookup[layer]["catalog"],
            schema_name=self.split_table_lookup[layer]["schema"],
            table_name=table_name,
            write_mode=load_type,
            update_condition=update_condition,
            delete_condition=delete_condition,
            merge_keys=self.key_columns,
            layer=layer,
            qualify_name=False,
            **(self._get_scd_type2_columns(schema_def) if schema_def and load_type == "scd_type_2" else dict()),
        )
        self.logger.info(f"{self.df.count()} records written to {layer}")

    @with_logging
    def _create_and_run_sql_file(self, schema_def: List[Dict[str, any]]) -> Tuple[str, str]:
        """
        This function generates a SQL file with the provided schema definition and runs it using dbutils.
        Args:
            schema_def (List[Dict[str, any]]): A list of dictionaries containing the schema definition of the delta table.
                Each dictionary should have the following keys:
                    - "name" (str): Field name.
                    - "type" (str): Data type.
                    - "nullable" (bool, optional): Indicates whether the field is nullable.
                    - "metadata" (Dict[str, str]): Optional metadata dictionary with comment.
        Returns:
            Tuple[str, str]: A tuple containing the filename and the generated SQL statement.
        """
        user_repo = self.dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
        filename = f"{self.schema.replace('_','')}_{self.name.replace('_','')}_create.sql"
        path = f"/Workspace/Repos/{user_repo}/naz-data/data/ddl/{self.catalog}/{self.schema}/{filename}"
        msg = f"{self.name} does not exist. Silver and gold objects must be created via explicit DDL. DDL has been generated in the following directory: {path}. The DDL has been ran to create the object in DEV. Please commit this file via a PR to have the object created in production"
        # call ddl generation function
        ddl = self.generate_uc_ddl(schema_def)
        # create the directory if it doesnt exist
        directory = os.path.dirname(path)
        if not os.path.exists(directory):
            os.makedirs(directory)
        # write the ddl file to the directory
        with open(path, "w") as f:
            f.write(ddl)
        # run sql that has been created
        self.spark.sql(ddl)
        # log warning informing caller that explicit ddl has been generated and ran to create the object
        self.logger.warning(msg)
        return filename, ddl

    @with_logging
    def generate_uc_ddl(self, schema_definition: List[dict]) -> str:
        """
        Generates a data definition language (DDL) statement to create a table in Unity Catalog.
        Args:
            schema_definition: A list of dictionaries containing the schema definition of the delta table to be exposed using the following formatting:
                "name": "<field_name>",
                "type": "<datatype>",
                "nullable": "<true/false>",
                "metadata": {"comment": "<field level comment>"}
        Returns:
            A string containing the DDL statement for creating the specified table in Unity Catalog.
        """
        template = DDL("UC_CREATE_TABLE")
        uc_create_stmnt = template(
            engine="databricks",
            catalog=f"{self.catalog}_dev",
            schema=self.schema,
            tablename=self.name,
            description=self.description,
            fields=schema_definition,
            keys=self.key_columns,
            replace=True,
        )
        return uc_create_stmnt

    @with_logging
    def archive(self, filename: str):
        """
        Archives the files in the medallion's raw directory.
        Parameters
        ----------
        None
        Returns
        -------
        None
        """
        archive_filename = f"{filename}_{dt.datetime.strftime(self.start_datetime, '%Y-%m-%dT%H%M%S')}"
        src_path = os.path.join(self.source_config["raw_path"], filename)
        tgt_path = os.path.join(self.source_config["raw_path"], "archive", archive_filename)

        self.dbutils.fs.mv(src_path, tgt_path)

    def compare_legacy_and_modern(
        self,
        dataset_name: str,
        legacy: str | DataFrame,
        modern: str | DataFrame,
        key_columns: List[str],
        completeness_lower_bound: float = 100.00,
        equivalency_lower_bound: float = 100.00,
        col_exclusions: Optional[List[str]] = None,
        auto_resolve_col_diffs: bool = False,
        treat_nulls: bool = False,
        drop_empty_cols: bool = False,
        comp_filter: str = "",
    ) -> None:
        """This method runs the completeness and equivalency checks to compare the legacy and modern versions of
        a dataset to ensure that they are the same. The goal is to ensure that we haven't changed the resulting
        data during our effort of modernizing the dataset.

        Args:
            dataset_name: The name of the dataset that we are comparing.
            legacy_query: Query to pull the legacy data from Snowflake.
            modern_query: Query to pull the modern data in ADLS that will replace the legacy Snowflake table.
            key_columns: The columns that make up the primary key of the table.
            completeness_lower_bound: Acceptable percentage for the two objects being compared. An completeness score >= to this value will succeed the check.
            equivalency_lower_bound: Acceptable percentage for the two objects being compared. An equivalency score >= to this value will succeed the check.
            col_exclusions: Columns that should be excluded in the comparison such as timestamp columns.
            auto_resolve_col_diffs: Whether to drop columns that appear in one DataFrame but not the other. Defaults to False.
            treat_nulls (bool): Whether to treat nulls and empty strings equal or as diffs. Defaults to False
            drop_empty_cols (bool): Whether to drop empty columns from final diff output so only columns with discrepancies are shown. Defaults to False and should be avoided for larger datasets/diffs.
            comp_filter (str): optional filter to be applied to modern and legacy dataframes before running checks
        Returns:
            None
        """
        # override if a value other than None is passed, reset to default values if None is passed
        completeness_lower_bound = 100.00 if completeness_lower_bound is None else completeness_lower_bound
        equivalency_lower_bound = 100.00 if equivalency_lower_bound is None else equivalency_lower_bound
        auto_resolve_col_diffs = False if auto_resolve_col_diffs is None else auto_resolve_col_diffs
        treat_nulls = False if treat_nulls is None else treat_nulls
        drop_empty_cols = False if drop_empty_cols is None else drop_empty_cols

        # make key columns lowercase since the checks make the column names lowercase
        key_columns = [c.lower() for c in key_columns]
        logger = self.logger
        # read modern
        logger.info("Reading legacy and modern datasets...")

        logger.info(f"Modern: {modern}")
        modern = modern.replace(self.catalog, f"{self.catalog}{MTU.get_catalog_environment_suffix()}")
        modern_df = modern if isinstance(modern, DataFrame) else self.spark.sql(modern)

        # read legacy
        try:
            logger.info(f"Legacy: {legacy}")
            if isinstance(legacy, DataFrame):
                legacy_df = legacy
            else:
                sf_extractor = SnowflakeExtractor(logger=logger)
                legacy_df = sf_extractor.query(legacy)
        except Exception as e:
            logger.warning(
                f"Unable to read legacy {dataset_name}. This could be caused by the legacy object being deprecated / inaccessible. Exception: {e}"
            )
            return  # intended behavior is to skip check so when legacy objects go offline, pipelines won't fail

        # apply any filters passed in
        if comp_filter:
            logger.info(f"Filtering legacy and modern on: {comp_filter}")
            modern = modern.filter(comp_filter)
            legacy = legacy.filter(comp_filter)

        logger.info("Running checks...")
        # run completeness check
        completeness_check = CompletenessIntegrityCheck(
            key_columns,
            dataset_name,
            on_failure="warning",
            lower_bound=completeness_lower_bound,
        )
        completeness_check.run(legacy_df, modern_df)

        # run equivalence check
        equivalency_check = EquivalencyIntegrityCheck(
            dataset_name,
            key_columns,
            on_failure="warning",
            lower_bound=equivalency_lower_bound,
            col_exclusions=col_exclusions,
            auto_resolve_col_diffs=auto_resolve_col_diffs,
            treat_nulls=treat_nulls,
        )
        diff = equivalency_check.run(legacy_df, modern_df)

        # if there are any differences, quarantine the data
        if diff.count() > 0:
            logger.info("Legacy and modern aren't equivalent, quarantining data for analysis...")
            Quarantine.quarantine_data(diff, dataset_name, "/mnt/alchemy-medallion")
        else:
            logger.info("Legacy and modern are both complete and equivalent.")
